In [4]:
!pip install numpy pandas matplotlib seaborn scikit-learn xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/101.7 MB 3.7 MB/s eta 0:00:27
    --------------------------------------- 1.8/101.7 MB 3.4 MB/s eta 0:00:30
   - -------------------------------------- 2.6/101.7 MB 3.5 MB/s eta 0:00:29
   - -------------------------------------- 3.1/101.7 MB 3.5 MB/s eta 0:00:29
   - -------------------------------------- 3.9/101.7 MB 3.5 MB/s eta 0:00:29
   - -------------------------------------- 4.7/101.7 MB 3.4 MB/s eta 0:00:29
   -- ------------------------------------- 5.2/101.7 MB 3.4 MB/s eta 0:00:29
   -- ------------------------------------- 6.0/101.7 MB 3.5 MB/s eta 0:00:28
   -- ------------------------------------- 6.8/101.7 MB 3.5 MB/s eta 0:00:28
   -- ------------------------------------- 7.6/101.7 MB 3.5 MB/s eta 0:00:28
   --- ------------------------------------ 8.4/101.7 MB 3.5 MB/s eta 0:00:27


In [ ]:

# ─────────────────────────────────────────────
# 5. CLASSICAL MODELS
# ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

models = {
    "Logistic Regression":     LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree":           DecisionTreeClassifier(random_state=42),
    "Random Forest":           RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting":       GradientBoostingClassifier(random_state=42),
    "XGBoost":                 XGBClassifier(eval_metric='mlogloss', random_state=42, verbosity=0),
    "SVM":                     SVC(probability=True, random_state=42),
    "KNN":                     KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes":             GaussianNB(),
}

results = {}
conf_matrices = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    t0 = time.time()
    cv_scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy')
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)

    acc   = accuracy_score(y_test, y_pred)
    f1    = f1_score(y_test, y_pred, average='weighted')
    auc   = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
    cv_m  = cv_scores.mean()
    cv_s  = cv_scores.std()

    results[name] = {
        'Accuracy': round(acc, 4),
        'F1 (weighted)': round(f1, 4),
        'AUC-ROC (OvR)': round(auc, 4),
        'CV Acc (mean)': round(cv_m, 4),
        'CV Acc (std)': round(cv_s, 4),
        'Train Time (s)': round(train_time, 3),
    }
    conf_matrices[name] = confusion_matrix(y_test, y_pred)
    print(f"  ✓ {name}: acc={acc:.3f} | f1={f1:.3f} | auc={auc:.3f} | time={train_time:.2f}s")

results_df = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'Model'})
print("\nResults DataFrame:")
print(results_df.to_string(index=False))

# ─────────────────────────────────────────────
# 6. RESULTS FIGURE — Figure 3
# ─────────────────────────────────────────────
fig3 = plt.figure(figsize=(22, 18))
fig3.suptitle("Model Performance — Classical Classifiers on OASIS Dataset",
              fontsize=17, fontweight='bold', y=0.99)
gs3 = gridspec.GridSpec(3, 3, figure=fig3, hspace=0.55, wspace=0.38)

model_names = list(results.keys())
colors_bar = plt.cm.tab10(np.linspace(0, 1, len(model_names)))

# 6a. Accuracy comparison
ax = fig3.add_subplot(gs3[0, 0])
acc_vals = [results[m]['Accuracy'] for m in model_names]
bars = ax.barh(model_names, acc_vals, color=colors_bar, edgecolor='white')
for bar, v in zip(bars, acc_vals):
    ax.text(v + 0.002, bar.get_y() + bar.get_height()/2, f'{v:.3f}', va='center', fontsize=9)
ax.set_xlim(0, 1.05)
ax.set_title("Test Accuracy", fontweight='bold')
ax.set_xlabel("Accuracy")

# 6b. F1 Score
ax = fig3.add_subplot(gs3[0, 1])
f1_vals = [results[m]['F1 (weighted)'] for m in model_names]
bars = ax.barh(model_names, f1_vals, color=colors_bar, edgecolor='white')
for bar, v in zip(bars, f1_vals):
    ax.text(v + 0.002, bar.get_y() + bar.get_height()/2, f'{v:.3f}', va='center', fontsize=9)
ax.set_xlim(0, 1.05)
ax.set_title("Weighted F1 Score", fontweight='bold')
ax.set_xlabel("F1")

# 6c. AUC-ROC
ax = fig3.add_subplot(gs3[0, 2])
auc_vals = [results[m]['AUC-ROC (OvR)'] for m in model_names]
bars = ax.barh(model_names, auc_vals, color=colors_bar, edgecolor='white')
for bar, v in zip(bars, auc_vals):
    ax.text(v + 0.002, bar.get_y() + bar.get_height()/2, f'{v:.3f}', va='center', fontsize=9)
ax.set_xlim(0, 1.08)
ax.set_title("AUC-ROC (OvR weighted)", fontweight='bold')
ax.set_xlabel("AUC")

# 6d. CV Accuracy with error bars
ax = fig3.add_subplot(gs3[1, 0])
cv_means = [results[m]['CV Acc (mean)'] for m in model_names]
cv_stds  = [results[m]['CV Acc (std)']  for m in model_names]
x_pos = np.arange(len(model_names))
ax.bar(x_pos, cv_means, yerr=cv_stds, capsize=5, color=colors_bar, edgecolor='white', alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels([m.replace(' ', '\n') for m in model_names], fontsize=8)
ax.set_title("5-Fold CV Accuracy ± Std", fontweight='bold')
ax.set_ylabel("CV Accuracy")
ax.set_ylim(0, 1.1)

# 6e. Training time
ax = fig3.add_subplot(gs3[1, 1])
times = [results[m]['Train Time (s)'] for m in model_names]
bars = ax.bar(x_pos, times, color=colors_bar, edgecolor='white', alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels([m.replace(' ', '\n') for m in model_names], fontsize=8)
ax.set_title("Training + CV Time (seconds)", fontweight='bold')
ax.set_ylabel("Seconds")

# 6f. Summary radar-like: top 3 models
ax = fig3.add_subplot(gs3[1, 2])
metrics_compare = ['Accuracy', 'F1 (weighted)', 'AUC-ROC (OvR)', 'CV Acc (mean)']
top3 = results_df.nlargest(3, 'Accuracy')['Model'].tolist()
x = np.arange(len(metrics_compare))
w = 0.25
for i, (m, col) in enumerate(zip(top3, ['#1E88E5', '#43A047', '#FB8C00'])):
    vals = [results[m][k] for k in metrics_compare]
    ax.bar(x + i*w, vals, w, label=m.replace(' ', '\n'), color=col, alpha=0.85, edgecolor='white')
ax.set_xticks(x + w)
ax.set_xticklabels(metrics_compare, fontsize=9, rotation=15)
ax.set_title("Top 3 Models — Metric Comparison", fontweight='bold')
ax.set_ylabel("Score")
ax.set_ylim(0, 1.15)
ax.legend(fontsize=8)

# 6g–6i: Confusion matrices for top 3 models
for i, m_name in enumerate(top3):
    ax = fig3.add_subplot(gs3[2, i])
    cm = conf_matrices[m_name]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, cbar=False, annot_kws={'size': 11})
    ax.set_title(f"Confusion Matrix\n{m_name}", fontweight='bold', fontsize=10)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.tick_params(labelsize=9)

plt.savefig(r'C:\Users\emade\Downloads\fig3_model_results.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 3 saved")

# ─────────────────────────────────────────────
# 7. FEATURE IMPORTANCE (Random Forest)
# ─────────────────────────────────────────────
rf_model = models['Random Forest']
fi = pd.Series(rf_model.feature_importances_, index=feature_labels[:len(FEATURES)]).sort_values()

fig4, axes4 = plt.subplots(1, 2, figsize=(16, 6))
fig4.suptitle("Feature Importance & Prediction Analysis", fontsize=15, fontweight='bold')

ax = axes4[0]
colors_fi = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(fi)))
ax.barh(fi.index, fi.values, color=colors_fi, edgecolor='white')
ax.set_title("Random Forest Feature Importances", fontweight='bold')
ax.set_xlabel("Importance")

# Prediction confidence distribution from Predictions.xlsx
ax = axes4[1]
pred_cols = ['confidence(Nondemented)', 'confidence(Demented)', 'confidence(Converted)']
pred_colors = [PALETTE['Nondemented'], PALETTE['Demented'], PALETTE['Converted']]
for col, col_color in zip(pred_cols, pred_colors):
    label = col.replace('confidence(', '').replace(')', '')
    ax.hist(pred[col], bins=20, alpha=0.65, color=col_color, label=label, edgecolor='white')
ax.set_title("Prediction Confidence Distribution\n(from Predictions.xlsx)", fontweight='bold')
ax.set_xlabel("Confidence"); ax.set_ylabel("Count")
ax.legend()

plt.tight_layout()
plt.savefig(r'C:\Users\emade\Downloads\fig4_feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Figure 4 saved")

# ─────────────────────────────────────────────
# 8. SAVE RESULTS TABLE to XLSX
# ─────────────────────────────────────────────
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

wb = Workbook()
ws = wb.active
ws.title = "Model Results"

headers = ['Model', 'Accuracy', 'F1 (weighted)', 'AUC-ROC (OvR)', 'CV Acc (mean)', 'CV Acc (std)', 'Train Time (s)']
header_fill = PatternFill('solid', start_color='1565C0')
header_font = Font(bold=True, color='FFFFFF', size=11)
thin = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)

ws.column_dimensions['A'].width = 22
for col_i in range(2, len(headers)+1):
    ws.column_dimensions[chr(64+col_i)].width = 16

for col_i, h in enumerate(headers, 1):
    cell = ws.cell(row=1, column=col_i, value=h)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal='center')
    cell.border = thin

row_fills = ['EEF2FF', 'FFFFFF']
for row_i, (_, row_data) in enumerate(results_df.iterrows(), 2):
    fill = PatternFill('solid', start_color=row_fills[row_i % 2])
    for col_i, val in enumerate(row_data, 1):
        cell = ws.cell(row=row_i, column=col_i, value=val)
        cell.fill = fill
        cell.alignment = Alignment(horizontal='center')
        cell.border = thin

wb.save(r'C:\Users\emade\Downloads\model_results.xlsx')
print("✓ Model results XLSX saved")

print("\n=== ALL DONE ===")

✓ Figure 1 saved
✓ Figure 2 saved
  ✓ Logistic Regression: acc=0.920 | f1=0.898 | auc=0.961 | time=0.21s
  ✓ Decision Tree: acc=0.840 | f1=0.830 | auc=0.865 | time=0.04s
  ✓ Random Forest: acc=0.920 | f1=0.898 | auc=0.948 | time=1.78s
  ✓ Gradient Boosting: acc=0.933 | f1=0.920 | auc=0.943 | time=4.94s
  ✓ XGBoost: acc=0.893 | f1=0.877 | auc=0.949 | time=2.88s
  ✓ SVM: acc=0.920 | f1=0.898 | auc=0.954 | time=0.24s
  ✓ KNN: acc=0.867 | f1=0.841 | auc=0.934 | time=0.08s
  ✓ Naive Bayes: acc=0.893 | f1=0.877 | auc=0.940 | time=0.02s

Results DataFrame:
              Model  Accuracy  F1 (weighted)  AUC-ROC (OvR)  CV Acc (mean)  CV Acc (std)  Train Time (s)
Logistic Regression    0.9200         0.8982         0.9606         0.9170        0.0174           0.207
      Decision Tree    0.8400         0.8300         0.8647         0.8472        0.0104           0.045
      Random Forest    0.9200         0.8982         0.9478         0.9169        0.0096           1.776
  Gradient Boosting    0